<a href="https://colab.research.google.com/github/pozdnyavladimer-jpg/geometric-state-navigator/blob/main/notebooks/gsl_multi_agent_mvp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


<a href="https://colab.research.google.com/github/pozdnyavladimer-jpg/geometric-state-navigator/blob/main/notebooks/gsl_multi_agent_mvp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GSL Multi-Agent MVP

This notebook demonstrates:

- simplified 19-node field state
- single-agent ranking
- coalition evaluation
- routed multi-step episode
- memory fold-in
- final field metrics

# Notebook Identity

## Name
GSL Multi-Agent MVP

## Type
Canonical demo

## Layer
Control layer

## Purpose
This notebook demonstrates how multiple agents propose candidate state transitions over a distributed field and how the system selects the coalition that best improves global state quality.

## Uses
- 6D state model
- single-agent evaluation
- coalition selection
- routed multi-step episode
- commit / soft commit / reroute logic
- memory fold-in

## State Model
This notebook uses the normalized 6D state:

[red_mass, orange_flow, yellow_struct, green_balance, blue_law, violet_future]

Where:
- red_mass = pressure / unresolved tension
- orange_flow = adaptability / movement
- yellow_struct = structural integrity
- green_balance = internal coherence
- blue_law = constraint compliance
- violet_future = transition readiness

## Input
Candidate deltas proposed by agents and coalitions.

## Output
This notebook should produce:
- ranked single agents
- ranked coalitions
- episode trace
- memory log
- final field metrics
- trend chart

## How to Run
Run all cells from top to bottom.

## How to Interpret Results
- COMMIT means an accepted state transition
- SOFT_COMMIT means conditionally accepted transition
- REROUTE means the current mode was not good enough and the system tries another mode
- REJECT means the proposed state was not accepted
- an empty memory log is possible if gating rules are too strict

## Safe Modifications
You can safely modify:
- agent deltas
- routing rules
- commit thresholds
- target weights
- plotting

## Do Not Change Casually
Do not modify without understanding why:
- state keys
- normalization rule
- metric definitions
- score composition

## Role in Repository
This notebook is a canonical entry demo for the control layer.
It is meant to be easier to read than research notebooks.

In [ ]:

# === GSL CORE (SIMPLIFIED) ===

import copy
import random

def normalize(v):
    s = sum(max(x, 0) for x in v.values())
    if s == 0:
        return {k: 0 for k in v}
    return {k: max(v[k], 0) / s for k in v}

def base_field():
    return [
        {
            "id": i,
            "state": normalize({
                "red_mass": 0.16,
                "orange_flow": 0.17,
                "yellow_struct": 0.17,
                "green_balance": 0.17,
                "blue_law": 0.16,
                "violet_future": 0.17
            })
        }
        for i in range(19)
    ]

def field_metrics(field):
    avg = {k: 0.0 for k in field[0]["state"]}

    for node in field:
        for k in avg:
            avg[k] += node["state"][k]

    for k in avg:
        avg[k] /= len(field)

    shadow = avg["red_mass"]
    coherence = 1 - (
        abs(avg["green_balance"] - avg["yellow_struct"]) +
        abs(avg["green_balance"] - avg["orange_flow"])
    )
    target_fit = avg["green_balance"] + avg["yellow_struct"]
    vitality = avg["orange_flow"] + avg["violet_future"]

    return {
        "shadow": round(shadow, 3),
        "coherence": round(coherence, 3),
        "target_fit": round(target_fit, 3),
        "vitality": round(vitality, 3)
    }

def apply_delta(field, delta):
    new_field = copy.deepcopy(field)

    for node in new_field:
        v = node["state"].copy()
        for k in v:
            v[k] += delta.get(k, 0.0)
        node["state"] = normalize(v)

    return new_field

def score_metrics(m):
    return round(
        m["coherence"] * 0.4
        - m["shadow"] * 0.3
        + m["target_fit"] * 0.2
        + m["vitality"] * 0.1,
        3
    )

In [ ]:

# === AGENTS ===

AGENTS = {
    "planner": {
        "text": "Propose structured plan with fallback paths.",
        "delta": {
            "yellow_struct": 0.10,
            "green_balance": 0.06,
            "orange_flow": 0.02,
            "red_mass": -0.04
        }
    },
    "critic": {
        "text": "Analyze risks and enforce constraints.",
        "delta": {
            "blue_law": 0.10,
            "yellow_struct": 0.05,
            "orange_flow": -0.04,
            "red_mass": -0.02
        }
    },
    "explorer": {
        "text": "Explore alternative strategies.",
        "delta": {
            "orange_flow": 0.12,
            "violet_future": 0.05,
            "yellow_struct": -0.04
        }
    },
    "stabilizer": {
        "text": "Reduce pressure and align system.",
        "delta": {
            "green_balance": 0.10,
            "red_mass": -0.08,
            "orange_flow": 0.03
        }
    }
}

In [ ]:

# === MULTI AGENT GSL ===

def evaluate_agents(field):
    results = {}

    for name, agent in AGENTS.items():
        new_field = apply_delta(field, agent["delta"])
        m = field_metrics(new_field)

        results[name] = {
            "metrics": m,
            "score": score_metrics(m),
            "text": agent["text"],
            "field": new_field,
            "delta": agent["delta"]
        }

    return results

def select_best(results):
    return max(results.items(), key=lambda x: x[1]["score"])


# === RUN ===

field = base_field()

results = evaluate_agents(field)
best = select_best(results)

print("=== MULTI-AGENT RANKING ===\n")

for name, data in sorted(results.items(), key=lambda x: -x[1]["score"]):
    print(
        f"{name:10} | score={data['score']} | "
        f"coh={data['metrics']['coherence']} | "
        f"shadow={data['metrics']['shadow']}"
    )
    print(" ", data["text"])
    print()

print("=== SELECTED AGENT ===")
print(best[0])
print(best[1]["text"])

=== MULTI-AGENT RANKING ===

planner    | score=0.46 | coh=0.93 | shadow=0.105
  Propose structured plan with fallback paths.

stabilizer | score=0.431 | coh=0.838 | shadow=0.076
  Reduce pressure and align system.

critic     | score=0.428 | coh=0.917 | shadow=0.128
  Analyze risks and enforce constraints.

explorer   | score=0.399 | coh=0.858 | shadow=0.142
  Explore alternative strategies.

=== SELECTED AGENT ===
planner
Propose structured plan with fallback paths.


In [ ]:

# === COALITION MODE (2 AGENTS TOGETHER) ===

from itertools import combinations

def merge_deltas(delta_a, delta_b):
    keys = set(delta_a.keys()) | set(delta_b.keys())
    return {k: delta_a.get(k, 0) + delta_b.get(k, 0) for k in keys}

def evaluate_coalitions(field):
    coalition_results = {}

    for a, b in combinations(AGENTS.keys(), 2):
        coalition_name = f"{a}+{b}"
        coalition_text = AGENTS[a]["text"] + " + " + AGENTS[b]["text"]

        merged_delta = merge_deltas(AGENTS[a]["delta"], AGENTS[b]["delta"])
        new_field = apply_delta(field, merged_delta)
        m = field_metrics(new_field)

        coalition_results[coalition_name] = {
            "agents": (a, b),
            "metrics": m,
            "score": score_metrics(m),
            "text": coalition_text,
            "delta": merged_delta,
            "field": new_field
        }

    return coalition_results

def select_best_coalition(results):
    return max(results.items(), key=lambda x: x[1]["score"])


# === RUN COALITIONS ===

coalition_results = evaluate_coalitions(field)
best_coalition = select_best_coalition(coalition_results)

print("=== COALITION RANKING ===\n")

for name, data in sorted(coalition_results.items(), key=lambda x: -x[1]["score"]):
    print(
        f"{name:20} | score={data['score']} | "
        f"coh={data['metrics']['coherence']} | "
        f"shadow={data['metrics']['shadow']} | "
        f"fit={data['metrics']['target_fit']} | "
        f"vit={data['metrics']['vitality']}"
    )
    print(" ", data["text"])
    print()

print("=== SELECTED COALITION ===")
print(best_coalition[0])
print(best_coalition[1]["text"])

print("\n=== MERGED DELTA ===")
for k, v in best_coalition[1]["delta"].items():
    print(f"{k:15}: {round(v, 3)}")

=== COALITION RANKING ===

planner+stabilizer   | score=0.466 | coh=0.857 | shadow=0.034 | fit=0.504 | vit=0.328
  Propose structured plan with fallback paths. + Reduce pressure and align system.

planner+explorer     | score=0.461 | coh=0.937 | shadow=0.094 | fit=0.362 | vit=0.417
  Propose structured plan with fallback paths. + Explore alternative strategies.

critic+stabilizer    | score=0.443 | coh=0.86 | shadow=0.053 | fit=0.43 | vit=0.289
  Analyze risks and enforce constraints. + Reduce pressure and align system.

planner+critic       | score=0.436 | coh=0.862 | shadow=0.081 | fit=0.447 | vit=0.26
  Propose structured plan with fallback paths. + Analyze risks and enforce constraints.

critic+explorer      | score=0.432 | coh=0.926 | shadow=0.115 | fit=0.287 | vit=0.385
  Analyze risks and enforce constraints. + Explore alternative strategies.

explorer+stabilizer  | score=0.429 | coh=0.839 | shadow=0.068 | fit=0.339 | vit=0.458
  Explore alternative strategies. + Reduce pressure

In [ ]:

# === ROUTING + GATING HELPERS ===

def route_mode(field):
    m = field_metrics(field)

    if m["shadow"] > 0.08:
        return "stabilize"
    elif m["target_fit"] < 0.52:
        return "plan"
    elif m["coherence"] < 0.90:
        return "explore"
    else:
        return "balanced"

def reroute_mode(mode):
    fallback = {
        "stabilize": "plan",
        "plan": "explore",
        "explore": "balanced",
        "balanced": "stabilize"
    }
    return fallback.get(mode, "balanced")

def evaluate_routed_coalitions(field, mode):
    all_results = evaluate_coalitions(field)
    routed = {}

    allowed = {
        "stabilize": [("planner", "stabilizer"), ("critic", "stabilizer")],
        "plan": [("planner", "critic"), ("planner", "stabilizer")],
        "explore": [("planner", "explorer"), ("critic", "explorer")],
        "balanced": [
            ("planner", "explorer"),
            ("planner", "stabilizer"),
            ("planner", "critic")
        ]
    }

    allowed_pairs = allowed.get(mode, [])

    for name, data in all_results.items():
        a, b = data["agents"]
        if (a, b) in allowed_pairs or (b, a) in allowed_pairs:
            routed[name] = data

    return routed

def gate_decision(metrics):
    shadow = metrics["shadow"]
    coh = metrics["coherence"]
    fit = metrics["target_fit"]
    vit = metrics["vitality"]

    if shadow <= 0.05 and coh >= 0.85:
        return "COMMIT"
    elif shadow <= 0.08 and coh >= 0.80:
        return "SOFT_COMMIT"
    elif vit >= 0.30:
        return "REROUTE"
    else:
        return "REJECT"

In [ ]:
import matplotlib.pyplot as plt